# RCA Pipeline Assignment

This notebook builds a retrieval-only RCA pipeline for the GeekShop alert clusters. It combines service graph traversal, temporal alert scoring, keyword kNN retrieval, and schema validation before writing `results/rca_output.json`, `FINDINGS.md`, and `SUBMIT.md`.

In [11]:
from pathlib import Path
import json
import math
import re
from collections import Counter, defaultdict
from datetime import datetime, timezone

try:
    import networkx as nx
except ModuleNotFoundError:
    class MiniDiGraph:
        def __init__(self):
            self._nodes = {}
            self._succ = defaultdict(dict)
            self._pred = defaultdict(dict)

        def add_node(self, name, **attrs):
            self._nodes.setdefault(name, {}).update(attrs)
            self._succ.setdefault(name, {})
            self._pred.setdefault(name, {})

        def add_edge(self, src, dst, **attrs):
            self.add_node(src)
            self.add_node(dst)
            self._succ[src][dst] = attrs
            self._pred[dst][src] = attrs

        def number_of_nodes(self):
            return len(self._nodes)

        def number_of_edges(self):
            return sum(len(v) for v in self._succ.values())

        def predecessors(self, node):
            return iter(self._pred.get(node, {}))

        def successors(self, node):
            return iter(self._succ.get(node, {}))

        @property
        def nodes(self):
            return self._nodes

        def __contains__(self, node):
            return node in self._nodes

    class MiniNX:
        DiGraph = MiniDiGraph

        @staticmethod
        def descendants(graph, node):
            seen, stack = set(), list(graph.successors(node))
            while stack:
                cur = stack.pop()
                if cur in seen:
                    continue
                seen.add(cur)
                stack.extend(graph.successors(cur))
            return seen

        @staticmethod
        def has_path(graph, src, dst):
            return src == dst or dst in MiniNX.descendants(graph, src)

        @staticmethod
        def shortest_path_length(graph, src, dst):
            if src == dst:
                return 0
            seen, queue = {src}, [(src, 0)]
            while queue:
                cur, dist = queue.pop(0)
                for nxt in graph.successors(cur):
                    if nxt == dst:
                        return dist + 1
                    if nxt not in seen:
                        seen.add(nxt)
                        queue.append((nxt, dist + 1))
            raise ValueError(f"No path from {src} to {dst}")

    nx = MiniNX()
    print("networkx not installed; using built-in MiniDiGraph fallback")

BASE = Path.cwd()
DATASET = BASE / "dataset"
RESULTS = BASE / "results"
RESULTS.mkdir(exist_ok=True)

cluster_path = RESULTS / "cluster_summary.json"
if not cluster_path.exists():
    fallback = BASE.parent / "d1" / "results" / "cluster_summary.json"
    if fallback.exists():
        cluster_path = fallback
    else:
        raise FileNotFoundError("Missing results/cluster_summary.json and ../d1/results/cluster_summary.json")

with cluster_path.open() as f:
    cluster_summary = json.load(f)
with (DATASET / "services.json").open() as f:
    service_map = json.load(f)
with (DATASET / "incidents_history.json").open() as f:
    incident_catalog = json.load(f)
with (DATASET / "alerts_sample.jsonl").open() as f:
    alerts = [json.loads(line) for line in f if line.strip()]

alerts_by_id = {a["id"]: a for a in alerts}
print(f"Loaded {len(cluster_summary['clusters'])} clusters, {len(alerts)} alerts, {len(incident_catalog['incidents'])} incidents")
print(f"Cluster input: {cluster_path}")

Loaded 5 clusters, 20 alerts, 29 incidents
Cluster input: /Users/trandinhthong/Downloads/AIOps/w2/d2/results/cluster_summary.json


In [12]:
G = nx.DiGraph()
criticality_weight = {"low": 0.15, "medium": 0.35, "high": 0.65, "critical": 0.9}

for service in service_map.get("services", []):
    G.add_node(service["name"], kind="service", **service)
for store in service_map.get("stores", []):
    G.add_node(store["name"], kind="store", **store)
for edge in service_map.get("edges", []):
    G.add_edge(edge["from"], edge["to"], **edge)

print(f"Graph nodes={G.number_of_nodes()} edges={G.number_of_edges()}")
print("Payment downstream:", list(nx.descendants(G, "payment-svc")))
print("Checkout neighbors:", sorted(set(G.predecessors("checkout-svc")) | set(G.successors("checkout-svc"))))

Graph nodes=14 edges=17
Payment downstream: ['payments-db']
Checkout neighbors: ['cart-svc', 'edge-lb', 'inventory-svc', 'notification-svc', 'payment-svc']


In [3]:
severity_weight = {"info": 0.05, "warn": 0.35, "high": 0.65, "crit": 1.0, "critical": 1.0}


def parse_ts(value):
    return datetime.fromisoformat(value.replace("Z", "+00:00"))


def normalize_metric(metric):
    return re.sub(r"[_\-]+", " ", metric.lower())


def token_set(text):
    return set(re.findall(r"[a-z0-9]+", text.lower()))


def alert_score(alert):
    sev = severity_weight.get(alert.get("severity", "warn"), 0.2)
    threshold = alert.get("threshold") or 0
    value = alert.get("value") or 0
    if isinstance(value, (int, float)) and isinstance(threshold, (int, float)) and threshold:
        ratio = min(max((value - threshold) / abs(threshold), 0), 2.0) / 2.0
    else:
        ratio = 0.0
    metric_boost = 0.25 if any(k in alert.get("metric", "") for k in ["connection", "error", "5xx", "drop", "queue"]) else 0.0
    return 0.55 * sev + 0.30 * ratio + metric_boost


def graph_temporal_score(cluster):
    cluster_alerts = [alerts_by_id[a] for a in cluster.get("alert_ids", []) if a in alerts_by_id]
    observed = set(cluster.get("services", []))
    if not cluster_alerts:
        observed = set(cluster.get("services", []))
    first_seen = {}
    direct_scores = defaultdict(float)
    for alert in cluster_alerts:
        service = alert["service"]
        ts = parse_ts(alert["ts"])
        first_seen[service] = min(first_seen.get(service, ts), ts)
        direct_scores[service] += alert_score(alert)

    candidates = set(observed)
    for service in list(observed):
        if service in G:
            candidates.update(G.predecessors(service))
            candidates.update(G.successors(service))

    min_ts = min(first_seen.values()) if first_seen else None
    raw = {}
    for candidate in candidates:
        if candidate not in G:
            continue
        node = G.nodes[candidate]
        score = direct_scores.get(candidate, 0.0)
        if candidate in observed:
            score += 0.45
        # Earlier symptoms are more likely to be causal than propagated alerts.
        if min_ts and candidate in first_seen:
            delay_s = max((first_seen[candidate] - min_ts).total_seconds(), 0)
            score += max(0.0, 0.35 - delay_s / 180.0)
        # Backing stores and critical services can be root causes even when they do not alert directly.
        score += criticality_weight.get(node.get("criticality", "medium"), 0.25) * 0.22
        if node.get("kind") == "store":
            score += 0.15
        # If a candidate is downstream of many observed services, it gets a causal boost.
        for observed_service in observed:
            if observed_service == candidate:
                continue
            if observed_service in G and candidate in G and nx.has_path(G, observed_service, candidate):
                distance = nx.shortest_path_length(G, observed_service, candidate)
                score += 0.28 / max(distance, 1)
            elif observed_service in G and candidate in G and nx.has_path(G, candidate, observed_service):
                distance = nx.shortest_path_length(G, candidate, observed_service)
                score += 0.10 / max(distance, 1)
        raw[candidate] = score

    max_score = max(raw.values()) if raw else 1.0
    ranked = sorted(((svc, round(score / max_score, 4)) for svc, score in raw.items()), key=lambda x: x[1], reverse=True)
    return ranked[:3]

for cluster in sorted(cluster_summary["clusters"], key=lambda c: c["alert_count"], reverse=True)[:3]:
    print(cluster["cluster_id"], graph_temporal_score(cluster))

c-000-000 [('payment-svc', 1.0), ('checkout-svc', 0.5547), ('edge-lb', 0.3785)]
c-000-001 [('notification-svc', 1.0), ('kafka-events', 0.2547), ('checkout-svc', 0.108)]
c-000-002 [('cart-svc', 1.0), ('cart-redis', 0.5131), ('catalog-svc', 0.3197)]


In [13]:
def incident_text(incident):
    return " ".join([
        incident.get("id", ""),
        incident.get("severity", ""),
        " ".join(incident.get("services_involved", [])),
        incident.get("root_cause_service", ""),
        incident.get("root_cause_class", ""),
        incident.get("summary", ""),
        incident.get("remediation", ""),
    ])


def cluster_text(cluster):
    metrics = []
    for fp in cluster.get("fingerprints", []):
        parts = fp.split("|")
        metrics.extend(parts)
    return " ".join([
        cluster.get("cluster_id", ""),
        cluster.get("max_severity", ""),
        " ".join(cluster.get("services", [])),
        " ".join(metrics),
    ])


def keyword_similarity(query, doc):
    q_tokens = token_set(query)
    d_tokens = token_set(doc)
    if not q_tokens or not d_tokens:
        return 0.0
    jaccard = len(q_tokens & d_tokens) / len(q_tokens | d_tokens)
    important = {"payment", "payments", "checkout", "cart", "redis", "kafka", "connection", "pool", "queue", "latency", "error", "db"}
    weighted_overlap = sum(2.4 if tok in important or tok.endswith("svc") else 1.0 for tok in q_tokens & d_tokens)
    phrase_boost = 0.0
    q_lower, d_lower = query.lower(), doc.lower()
    for phrase in ["connection pool", "db connection", "queue lag", "queue depth", "cart redis", "payment timeout"]:
        phrase_tokens = phrase.split()
        if all(tok in q_lower for tok in phrase_tokens) and all(tok in d_lower for tok in phrase_tokens):
            phrase_boost += 0.22
    return jaccard + weighted_overlap / (len(q_tokens) + 8) + phrase_boost


def retrieve_similar(cluster, top_k=3):
    query = cluster_text(cluster)
    cluster_services = set(cluster.get("services", []))
    scored = []
    for incident in incident_catalog["incidents"]:
        sim = keyword_similarity(query, incident_text(incident))
        incident_services = set(incident.get("services_involved", []))
        sim += 0.16 * len(cluster_services & incident_services)
        if incident.get("root_cause_service") in cluster_services:
            sim += 0.08
        if sim > 0:
            scored.append((incident, sim))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

for cluster in sorted(cluster_summary["clusters"], key=lambda c: c["alert_count"], reverse=True)[:3]:
    similar = retrieve_similar(cluster, top_k=3)
    print(cluster["cluster_id"], [(inc["id"], inc["root_cause_class"], round(score, 3)) for inc, score in similar])

c-000-000 [('INC-2025-11-08', 'connection_pool_exhaustion', 1.49), ('INC-2025-09-05', 'connection_pool_exhaustion', 1.155), ('INC-2026-05-10', 'connection_pool_exhaustion', 1.134)]
c-000-001 [('INC-2026-02-08', 'downstream_provider', 0.633), ('INC-2025-10-03', 'rebalance_storm', 0.542), ('INC-2026-05-18', 'feature_flag', 0.476)]
c-000-002 [('INC-2026-02-22', 'network_partition', 0.501), ('INC-2025-07-19', 'eviction', 0.494), ('INC-2026-04-02', 'deadlock', 0.438)]


In [14]:
def split_actions(remediation):
    parts = re.split(r"\.\s+|;\s+", remediation.strip())
    return [p.strip().rstrip(".") for p in parts if p.strip()]


def analyze_cluster(cluster):
    graph_top3 = graph_temporal_score(cluster)
    similar = retrieve_similar(cluster, top_k=3)
    if similar:
        top_incident, similarity = similar[0]
        retrieved_root = top_incident.get("root_cause_service")
        graph_services = [service for service, _ in graph_top3]
        root_cause = retrieved_root if retrieved_root in graph_services else graph_top3[0][0]
        root_class = top_incident.get("root_cause_class", "unknown")
        actions = split_actions(top_incident.get("remediation", "Escalate to service owner"))
        similar_ids = [inc["id"] for inc, _ in similar]
        evidence = min(1.0, math.log1p(cluster.get("alert_count", 1)) / math.log1p(15))
        sim_strength = min(1.0, similarity / 1.75)
        confidence = min(0.93, 0.34 + 0.25 * graph_top3[0][1] + 0.22 * sim_strength + 0.12 * evidence)
        reasoning = (
            f"Graph traversal ranked {graph_top3[0][0]} highest from temporal order, severity, topology, and neighboring dependencies. "
            f"Keyword kNN retrieved {top_incident['id']} as nearest history item with class {root_class}; "
            f"root service chosen as {root_cause} after reconciling graph candidates with retrieved incident."
        )
    else:
        root_cause = graph_top3[0][0] if graph_top3 else cluster.get("services", ["unknown"])[0]
        root_class = "unknown"
        actions = ["Escalate to owning service team", "Collect traces and recent deploy events"]
        similar_ids = []
        confidence = 0.35
        reasoning = "No similar incident was retrieved, so RCA falls back to graph and temporal scoring only."
    return {
        "cluster_id": cluster["cluster_id"],
        "graph_top3": [[svc, round(float(score), 2)] for svc, score in graph_top3],
        "root_cause": root_cause,
        "class": root_class,
        "confidence": round(float(confidence), 2),
        "actions": actions[:3],
        "reasoning": reasoning,
        "similar_incidents": similar_ids,
        "method": "graph+retrieval",
    }

clusters_to_analyze = sorted(cluster_summary["clusters"], key=lambda c: c["alert_count"], reverse=True)[:3]
results = [analyze_cluster(cluster) for cluster in clusters_to_analyze]
rca_output = {"clusters_analyzed": len(results), "results": results}

print(json.dumps(rca_output, indent=2, ensure_ascii=False))

{
  "clusters_analyzed": 3,
  "results": [
    {
      "cluster_id": "c-000-000",
      "graph_top3": [
        [
          "payment-svc",
          1.0
        ],
        [
          "checkout-svc",
          0.55
        ],
        [
          "edge-lb",
          0.38
        ]
      ],
      "root_cause": "payment-svc",
      "class": "connection_pool_exhaustion",
      "confidence": 0.9,
      "actions": [
        "Rollback to v3.1",
        "Scale pool 50 → 100 cushion",
        "Add pool monitor alert > 80%"
      ],
      "reasoning": "Graph traversal ranked payment-svc highest from temporal order, severity, topology, and neighboring dependencies. Keyword kNN retrieved INC-2025-11-08 as nearest history item with class connection_pool_exhaustion; root service chosen as payment-svc after reconciling graph candidates with retrieved incident.",
      "similar_incidents": [
        "INC-2025-11-08",
        "INC-2025-09-05",
        "INC-2026-05-10"
      ],
      "method": "graph+r

In [15]:
required_result_keys = {"cluster_id", "graph_top3", "root_cause", "class", "confidence", "actions", "reasoning", "similar_incidents", "method"}
assert rca_output["clusters_analyzed"] == len(rca_output["results"]) >= 3
for item in rca_output["results"]:
    missing = required_result_keys - set(item)
    assert not missing, missing
    assert item["graph_top3"] and len(item["graph_top3"]) <= 3
    assert isinstance(item["root_cause"], str) and item["root_cause"]
    assert isinstance(item["class"], str) and item["class"]
    assert 0 <= item["confidence"] <= 1
    assert isinstance(item["actions"], list) and item["actions"]
    assert isinstance(item["similar_incidents"], list)

with (RESULTS / "rca_output.json").open("w") as f:
    json.dump(rca_output, f, indent=2, ensure_ascii=False)

print("Wrote", RESULTS / "rca_output.json")
print("Validated RCA output schema for", rca_output["clusters_analyzed"], "clusters")

Wrote /Users/trandinhthong/Downloads/AIOps/w2/d2/results/rca_output.json
Validated RCA output schema for 3 clusters


In [16]:
main = results[0]
uncertain = min(results, key=lambda item: item["confidence"])
threshold = 0.95
findings = f"""# FINDINGS.md

## RCA findings

Cluster chính là `{main['cluster_id']}` với {clusters_to_analyze[0]['alert_count']} alerts, trải trên `payment-svc`, `checkout-svc`, và `edge-lb`. RCA pipeline chọn root cause là `{main['root_cause']}` với class `{main['class']}` và confidence `{main['confidence']}`. Lý do cụ thể: alert đầu tiên xuất hiện ở `payment-svc` lúc 09:42 với `db_connection_pool_used_ratio` tăng từ warn lên crit, sau đó mới lan sang latency/error ở checkout và 5xx ở edge-lb. Graph cũng ủng hộ hướng này vì checkout gọi payment, còn edge-lb là entry point nên khả năng cao chỉ nhìn thấy propagation. Retrieval top-3 gồm {', '.join(main['similar_incidents'])}; incident gần nhất mô tả đúng pattern DB pool leak và cascade checkout.

Với confidence `{main['confidence']}`, mình chưa dám deploy auto-remediation kiểu rollback hoàn toàn tự động. Nếu phải đặt threshold cho auto-rollback không cần SRE confirm, mình chọn `{threshold}`: cao hơn output hiện tại và đủ cao để chỉ chạy khi graph score và retrieval đều đồng thuận rất mạnh. Case không chắc nhất là `{uncertain['cluster_id']}`: root cause `{uncertain['root_cause']}`, class `{uncertain['class']}`, confidence `{uncertain['confidence']}`. Cluster này ít alert hơn nên bằng chứng temporal mỏng; retrieval có thể đang match theo service/metric keyword thay vì cùng cơ chế lỗi thật.

Mình không chọn bonus path vì yêu cầu chính đã đủ với retrieval-only: service map ổn định, incident history có nhiều pattern e-commerce lặp lại, và classifier kNN top-1 trả được class + action có thể audit được. Decision tree trên 30 incident dễ overfit, TF-IDF sẽ tốt hơn keyword nhưng chưa cần cho dataset nhỏ, còn LLM enrichment thêm chi phí và biến thiên trong khi không cần API key cho bài này.
"""

submit = f"""# SUBMIT.md

## EOD Checkpoint

1. Confidence của top-1 trong cluster lớn nhất `{main['cluster_id']}` là `{main['confidence']}`. Nếu phải set threshold để auto-rollback không cần SRE confirm, mình chọn `{threshold}`. Lý do: cluster lớn có tín hiệu rất mạnh từ temporal order (`payment-svc` báo pool trước), graph dependency (checkout gọi payment, edge chỉ thấy downstream 5xx), và retrieval khớp các incident pool exhaustion cũ. Dù vậy rollback tự động là hành động rủi ro nên mình muốn threshold cao hơn confidence hiện tại, trừ khi có thêm deploy-event evidence.

2. Variant classifier mình chọn là A rule-based/kNN retrieval. Chạy thực tế ổn: top-1 incident cung cấp `root_cause_class`, remediation được tách thành action list, và fallback graph-only vẫn tồn tại khi retrieval rỗng. Trade-off với LLM là kNN ít linh hoạt hơn trong diễn giải ngôn ngữ, nhưng deterministic, không cần API key, dễ debug, và phù hợp auto-grader. Paid/free LLM có thể viết reasoning tốt hơn nhưng tăng latency, chi phí, và rủi ro output lệch schema.

3. Pipeline này gần nhóm product event-correlation/RCA dựa trên topology + historical retrieval nhất, giống hướng Moogsoft/BigPanda/Dynatrace hơn là một chatbot LLM thuần. Với GeekShop, lựa chọn đó hợp lý vì alert volume cao, service map tương đối ổn định, và các incident lặp lại theo pattern như connection pool, queue lag, cache/DB contention. Mình sẽ chỉ đổi sang LLM-heavy khi incident history ít cấu trúc hơn hoặc cần tổng hợp logs/traces dạng text dài.
"""

(BASE / "FINDINGS.md").write_text(findings, encoding="utf-8")
(BASE / "SUBMIT.md").write_text(submit, encoding="utf-8")
print("Wrote FINDINGS.md and SUBMIT.md")
word_count = len(re.findall(r"\w+", findings))
print(f"FINDINGS words: {word_count}")

Wrote FINDINGS.md and SUBMIT.md
FINDINGS words: 298
